# **Important Technical Details:**

<table>
  <tr><th>Topic</th><th>What you should be able to explain</th></tr>
  <tr><td>Execution plans</td><td>Scans vs. seeks, joins, estimated vs. actual rows, key lookups, spills</td></tr>
  <tr><td>Statistics</td><td>How SQL Server estimates cardinality and chooses query plans</td></tr>
  <tr><td>Parameter sniffing</td><td>How parameter values at compilation can produce a plan that performs poorly for other values</td></tr>
  <tr><td>Isolation levels</td><td>Read Committed, Snapshot, and Serializable trade-offs</td></tr>
  <tr><td>Query Store</td><td>Query history, plan changes, and performance regression analysis</td></tr>
  <tr><td>Deadlocks</td><td>Deadlock graph, victim selection, consistent lock order, short transactions</td></tr>
  <tr><td>Tempdb</td><td>Temporary objects, spills, allocation contention, capacity</td></tr>
  <tr><td>Transaction log</td><td>Log backups, log reuse, recovery models, long-running transactions</td></tr>
  <tr><td>Backup verification</td><td>Backup checks, restore tests, recovery chains, encryption keys</td></tr>
  <tr><td>HR/DR</td><td>Availability Groups, FCIs, replication, RPO, RTO, planned and unplanned failover</td></tr>
  <tr><td>Security</td><td>Least privilege, roles, auditing, TLS, TDE, backup protection</td></tr>
  <tr><td>Deployment</td><td>Source control, schema comparison, migration testing, rollback and recovery</td></tr>
  <tr><td>Capacity</td><td>MainData/log growth, file sizing, autogrowth, storage forecastingtains</td></tr>
</table>

# **Troubleshooting**

## **How do you troubleshoot a slow SQL Server query?**

- I first establish the scope and impact: whether the problem affects one query, one application, or the entire SQL Server instance. I compare current performance with the normal baseline, inspect the actual execution plan, and check logical reads, CPU time, elapsed time, waits, blocking, and recent changes. Then I identify the root cause before changing indexes, statistics, the query, or server configuration. After implementing a fix, I validate the improvement and monitor for regressions.

**Use this troubleshooting sequence:**
1. **Establish the scope**
    - One query, one database, or the whole instance? When did the issue start? What changed?
2. **Inspect execution and resource usage**
    - Actual execution plan, Query Store, logical reads, CPU, duration, waits, and memory grants.
3. **Identify the bottleneck**
    - Scans, inefficient joins, stale statistics, parameter sniffing, blocking, storage latency, or CPU pressure.
4. **Make a targeted change**
    - Rewrite the query, adjust an index, update statistics, address contention, or resolve an infrastructure issue.
5. **Validate and prevent recurrence**
    - Compare before/after performance, verify correctness, document the root cause, and add monitoring.

**Useful tools to mention:** Query Store, actual execution plans, SET STATISTICS IO, TIME ON, Extended Events, DMVs, and SQL Server monitoring tools.

## **A stored procedure that normally runs in seconds now takes several minutes. How would you troubleshoot it?**

I would start by establishing the scope of the regression and identifying what changed. I would compare recent and historical execution behavior using Query Store where available, then inspect the actual execution plan and runtime metrics.

I would look for issues such as scans on large tables, inaccurate estimates, parameter-sensitive plans, implicit conversions, blocking, and changes in data volume or statistics.

Based on the evidence, I would make the smallest effective change, such as improving an index, rewriting a predicate, correcting a join, or addressing parameter sensitivity. I would test with representative parameter values, compare logical reads and elapsed time, validate the results, and monitor production after deployment.

I would also document the root cause so the fix is repeatable and the team can recognize the same pattern in the future.

<table>
<td>

1. **Establish the scope**
    - Determine whether every execution is slow or only certain parameters, users, or time periods. Check when the problem started and what changed.
2. **Inspect runtime evidence**
    - Review Query Store, actual execution plans, logical reads, CPU time, duration, waits, blocking, and relevant database metrics.
3. **Identify the bottleneck**
    - Look for table scans, excessive key lookups, inaccurate cardinality estimates, parameter-sensitive plans, implicit conversions, expensive joins, or resource contention.
4. **Apply a measured fix**
    - Consider indexing, query rewrites, updated statistics, appropriate parameter handling, or resolving blocking. Avoid speculative index creation or forcing plans without evidence.
5. **Validate and monitor**
    - Compare before-and-after execution time, reads, CPU, and correctness. Test representative parameters and monitor for regressions.

</td>
</table>

## **<span style = "color:Red">Scenario:</span> A production application suddenly becomes slow. The application was working normally this morning. Now users say every page is slow. What do you do?**

<table>
    <tr><td>First, I <strong>establish impact</strong> and severity with the application team. I determine whether the issue affects one database, multiple databases, or the entire SQL Server instance. I <strong>check current resource utilization, active requests, blocking, waits, storage latency, and recent deployments or configuration changes.</strong> If a specific query is responsible, I inspect its <strong>execution plan</strong> and <strong>Query Store history</strong>. If the issue is <strong>blocking</strong>, I investigate the <strong>head blocker</strong> and transaction. If CPU, memory, or I/O is saturated, I determine which workload is responsible before changing configuration. I <strong>communicate status</strong> and workarounds, implement the safest targeted mitigation, validate recovery, and follow up with a root-cause analysis and preventive actions.</td></tr>
</table>

## **<span style = "color:red">Scenario:</span> Database storage is growing rapidly. A production database is running out of disk space. What do you do?**

Explain that you would:
- Identify whether the growth is in data files, transaction logs, tempdb, or backup storage.
- Check free disk capacity, database file growth settings, log reuse wait, and recent workload changes.
- Determine whether a long-running transaction, stalled log backup, bulk load, or unusual data growth is responsible.
- Provide safe additional capacity if needed, then resolve the underlying cause.
- Review growth trends and forecast capacity.

Avoid casually shrinking database files as a routine fix. Shrinking can create fragmentation and repeated regrowth overhead.

# **Indexing**

- Indexes consume storage and add overhead to inserts, updates, and deletes. I design indexes around actual query patterns, selectivity, workload, and existing indexes, then validate the benefit using execution plans and workload measurements.

## **What is the difference between a clustered and a nonclustered index?**

<table>
  <tr><th>Clustered Index</th><th>Nonclustered Index</th></tr>
  <tr><td>Determines the logical ordering of rows in the index structure</td><td>Maintains a separate index structure containing keys and row locators</td></tr>
  <tr><td>Only one per table</td><td>Multiple per table</td></tr>
  <tr><td>Leaf level contains the table's data rows</td><td>Leaf level contains index keys and included columns</td></tr>
  <tr><td>Useful for access patterns aligned with the clustered key</td><td>Useful for selective lookups, joins, and covering queries</td></tr>
</table>

<table>
  <tr><th> </th><th>Clustered Index</th><th>Nonclustered Index</th></tr>
  <tr><td>Definition</td><td>Pysicaly sorts and stores datarows at B-Tree</td><td>Separate structure with pointers to the data</td></tr>
  <tr><td>Number of indexes</td><td>One index per table</td><td>Multiple indexes are allowed</td></tr>
  <tr><td>Read performance</td><td>Faster</td><td>slower</td></tr>
  <tr><td>Write performance</td><td>Slower, due to potentioal data row reordering</td><td>Faster, since physical data order is unaffected</td></tr>
  <tr><td>Storage efficiency</td><td>more storage-efficient</td><td>Requires additional storage space</td></tr>
  <tr><td>Use cases</td><td>Unique column, Not frequently modified column, Improve range query performance</td><td>Columns frequently used in search conditions and joins, Exact atch queries</td></tr>
</table>

### **Example**

In [ ]:
CREATE NONCLUSTERED INDEX ix_Orders_store_status_date
ON Sales.Orders(store_id, status, order_date DESC)
INCLUDE (total_amount);

- This builds a separate **B-tree** on Sales.Orders, sorted for one specific query shape: “filter by store and status, newest first, show the total.”

**Line 1: CREATE NONCLUSTERED INDEX ix_Orders_store_status_date**
- CREATE NONCLUSTERED INDEX says to build a separate lookup structure alongside the table. It does not reorder the table itself (that would be a clustered index, of which there can be only one).
- ix_Orders_store_status_date is the index name. It must be unique per table. The convention ix_<Table>_<key columns> makes the purpose readable in execution plans and in sys.indexes. (ix_ is for ordinary indexes, and some teams use ux_ for unique ones, as in the earlier soft-delete example.)

**Line 2: ON Sales.Orders(store_id, status, order_date DESC)**
- ON Sales.Orders names the table being indexed.
- (store_id, status, order_date DESC) are the **key columns**. They define the index’s sort order: rows are sorted by **store_id first**, then by **status** within each store, then by **order_date** newest-first within each store and status.
    - <span style = "color:red">**Why this column order matters**</span>. An index is useful starting from its leftmost column, like a phone book sorted by last name, then first name:

        <table>
            <tr><th>Query filters on</th><th>Can seek?</th></tr>
            <tr><td>store_id</td><td>Yes</tr>
            <tr><td>store_id and status</td><td>Yes</tr>
            <tr><td>store_id, status, and a range on order_date</td><td>Yes, best case</tr>
            <tr><td>store_id and order_date (skipping status)</td><td>Partly: seeks on store_id, then must scan within it</tr>
            <tr><td>status only</td><td>No, it can’t jump straight in, since the leading column isn’t constrained</tr>
            <tr><td>order_date only</td><td>No</tr>
        </table>

<table>
    <tr><td>The general rule for choosing order: equality columns first (store_id = 7, status = 'Pending'), then the sort/range column last (order_date). Within the equality group, put the more selective column first when you can. Here store_id leads because it narrows the data most.</td></tr>
<table>

**Line 3: INCLUDE (total_amount)**
- INCLUDE columns are stored at the leaf level only. They’re carried along as payload, but they are not part of the sort order and can’t be used to seek.
- total_amount is there because the query returns it. Without it, the engine would find each matching row in the index and then do a key lookup back to the table just to read that one column.

**Why INCLUDE instead of adding it to the key?**
- Key columns appear at every level of the B-tree, so a wider key makes the whole tree larger and deeper. Included columns live only in the leaves.
- Key columns have size limits (about 1,700 bytes for nonclustered keys in SQL Server), while included columns are more flexible.
- Since nothing filters or sorts on total_amount, putting it in the key would add cost and no benefit.

**What you get for free: <span style = "color:red">order_id</span>**
- Every nonclustered index silently carries the table’s clustering key as its row locator. If order_id is the clustered key on Sales.Orders, then order_id is available in this index too, even though it isn’t listed. That’s why the query below can return it without a lookup.

In [ ]:
SELECT TOP 20 order_id, order_date, total_amount
FROM Sales.Orders
WHERE store_id = 7 AND status = 'Pending'
ORDER BY order_date DESC;

<table>
  <tr><th>Query element</th><th>Satisfied by</th></tr>
  <tr><td>WHERE store_id = 7</td><td>Key column 1: seek</tr>
  <tr><td>AND status = 'Pending'</td><td>Key column 2: seek</tr>
  <tr><td>ORDER BY order_date DESC</td><td>Key column 3: already sorted</tr>
  <tr><td>SELECT order_id</td><td>Clustering key, carried automatically</tr>
  <tr><td>SELECT order_date</td><td>Key column 3</tr>
  <tr><td>SELECT total_amount</td><td>INCLUDE column</tr>
</table>

- Every column the query touches is in the index, so it’s <span style = "color:green">**covering**</span>: the execution plan shows a single Index Seek with no Key Lookup and no Sort. The engine reads 20 adjacent index entries and stops.

**The costs**
- **Writes:** every INSERT into Sales.Orders also inserts into this index. An UPDATE to store_id, status, order_date, or total_amount also updates it. Moving an order from Pending to Shipped relocates its entry in the index.
- **Storage:** the index stores copies of four columns plus the clustering key for every row.
- **Maintenance:** like any index, it can fragment over time and needs fresh statistics.

**Options you’ll often see added**

In [ ]:
CREATE NONCLUSTERED INDEX ix_Orders_store_status_date
ON Sales.Orders(store_id, status, order_date DESC)
INCLUDE (total_amount)
WHERE status = 'Pending'                -- filtered index: only index Pending rows
WITH (ONLINE = ON, FILLFACTOR = 90);    -- build without blocking writes; leave 10% free space per page

- **Filtered index (<span style = "color:orange">WHERE</span> ...):** if the app only ever queries pending orders, index just those. It’s far smaller and cheaper to maintain. (The status key column is then redundant and could be dropped.)
- <span style = "color:orange">**ONLINE = ON:**</span> builds the index while the table stays available (an Enterprise/Azure SQL feature). Without it, a build on a 20-million-row table can block writers.
- <span style = "color:orange">**FILLFACTOR:**</span> leaves free space on pages to reduce splits when inserts land in the middle of the index.

## **<span style = "color:Red">Scenario:</span> A developer wants an index created immediately. The developer says an index will fix the performance problem and wants you to create it in production right now.**

<table>
<tr><td>I would ask for the affected query and the evidence supporting the proposed index. I would review the execution plan, existing indexes, selectivity, and write workload. If the index is justified, I would assess its storage, locking, and deployment impact and use the approved change process. For a critical incident, I would follow the emergency-change process rather than bypassing change control entirely. After deployment, I would measure the results.</td></tr>
</table>

# **Blocking and Deadlocks** 

## **How do you identify and resolve blocking and deadlocks?**

- <span style = "color:green">**Blocking:**</span> One session waits because another session holds an incompatible lock.
- <span style = "color:green">**Deadlock:**</span> Two or more sessions form a cycle of dependencies; SQL Server chooses a victim and rolls back that transaction.

<table>
    <tr><td>I first distinguish blocking from a deadlock. For blocking, I identify the head blocker, the waiting sessions, the statements involved, and how long the blocking has continued. I investigate transaction duration, isolation levels, access patterns, and missing indexes. For deadlocks, I inspect the deadlock graph captured by Extended Events to understand the resources and lock order involved. I then fix the underlying cause, such as inconsistent access order, excessive transaction scope, or an inefficient query. I avoid killing sessions without understanding the impact</td></tr>
</table>

In [ ]:
SELECT
    session_id,
    blocking_session_id,
    wait_type,
    wait_time,
    wait_resource,
    status
FROM sys.dm_exec_requests
WHERE blocking_session_id <> 0;

# **Backup & Recovery**

## **How do you design a backup and recovery strategy?**

Explain these concepts:
- **Full backup:** Backs up the database's data.
- **Differential backup:** Captures changes since the most recent full backup.
- **Transaction log backup:** Captures log records since the previous log backup, for databases using the Full or Bulk-logged recovery model.
- **RPO (Recovery Point Objective):** How much data loss the business can tolerate, expressed as time.
- **RTO (Recovery Time Objective):** How long the business can tolerate the system being unavailable.

<table>
    <tr><td>I begin with business-defined RPO and RTO requirements. I establish an appropriate full, differential, and transaction-log backup schedule based on the recovery model and workload, protect backups from unauthorized access, monitor job success and backup age, and test restores regularly. I verify the full recovery chain and document recovery procedures. I also distinguish high availability from disaster recovery: redundancy reduces downtime, while backups provide a separate recovery path from corruption, accidental deletion, and other failures</td></tr>
</table>

## **<span style = "color:red">Scenario:</span> A backup job succeeds, but a restore fails. How can this happen, and what would you do?**

A backup job completing successfully does not prove that the entire recovery process will work.
- <span style = "color:red">**Potential causes:**</span> Missing transaction-log backups, an incomplete backup chain, unavailable encryption certificates or keys, storage problems, or an invalid recovery sequence.

<table>
    <tr><td>I would identify the required <span style="color: orange; font-weight: bold;">recovery point</span>, preserve the available backup files, review <span style="color: orange; font-weight: bold;">backup history</span> and <span style="color: orange; font-weight: bold;">error logs</span>, and validate the <span style="color: orange; font-weight: bold;">full/differential/log sequence</span>. I would restore to an <span style="color: orange; font-weight: bold;">isolated environment</span> where possible, verify database recovery and application-critical data, and <span style="color: orange; font-weight: bold;">communicate any gap</span> against the recovery objectives. I would then correct the backup process and <span style="color: orange; font-weight: bold;">establish recurring restore tests</span>.</td></tr>
</table>

# **HA/DR**

## **Explain Always On Availability Groups versus failover clustering.**

<table>
    <tr><th>Feature</th><th>Always On Availability Groups</th><th>Failover Cluster Instance (FCI)</th></tr>
    <tr><td>Primary purpose</td><td>Database-level availability</td><td>Instance-level availability</td></tr>
    <tr><td>Replication/storage</td><td>Maintains database replicas, typically using separate database copies</td><td>Uses shared storage or an equivalent supported clustered storage design</td></tr>
    <tr><td>Failover scope</td><td>Databases in the availability group</td><td>SQL Server instance</td></tr>
    <tr><td>Typical use</td><td>Protecting selected databases and enabling eligible secondary workloads</td><td>Protecting the SQL Server instance and its databases</td></tr>
</table>

- An Availability Group protects the databases in the group; an FCI protects the SQL Server instance. Both rely on Windows Server Failover Clustering for the relevant supported configurations, but they solve different availability problems.

<table>
    <tr><td>I would select the architecture based on the required availability, recovery objectives, storage design, licensing, network topology, and operational complexity. I would also validate failover behavior, client reconnection, monitoring, backup configuration, and recovery procedures. I would not assume that having a secondary replica automatically means zero data loss or that every failure triggers a successful automatic failover.</td></tr>
</table>

## **<span style = "color:Red">Scenario:</span> A database is unavailable after a server failure. The primary SQL Server has failed. What do you do?**

Your response should cover:
1. Confirm the failure and assess the business impact.
2. Determine whether an approved automatic failover occurred.
3. Verify the health and synchronization state of the intended secondary replica or clustered instance.
4. Follow the documented failover procedure and verify application connectivity.
5. Validate database consistency and critical application functions.
6. Communicate recovery status and document the incident.

Do not immediately force failover without assessing the state of the replicas. A forced failover can involve data loss depending on the configuration and circumstances.

# **Security**

Mention the principle of <span style = "color:green">**least privilege**</span> and <span style = "color:green">**layered security**</span>.
- Use Windows or Microsoft Entra <span style = "color:orange">**authentication**</span> where appropriate, and controlled SQL <span style = "color:orange">**logins**</span> where required.
- Grant permissions through <span style = "color:orange">**roles**</span> rather than granting broad permissions directly to individuals.
- Separate administrative, application, developer, and reporting access.
- Apply <span style = "color:orange">**encryption**</span> in transit using <span style = "color:orange">**Transport Layer Security (TLS)**</span> and encryption at rest, such as <span style = "color:orange">**Transparent Data Encryption (TDE)**</span>, according to the security requirements.
- Evaluate <span style = "color:orange">**Always Encrypted**</span> or <span style = "color:orange">**column-level protection**</span> when the threat model requires it.
- <span style = "color:orange">**Audit**</span> access to sensitive data and privileged actions.
- Protect backup files, secrets, and deployment <span style = "color:orange">**credentials**</span>.
- Avoid exposing sensitive records in logs, diagnostic output, or developer environments.

In [ ]:
CREATE ROLE ReportingReader;

GRANT SELECT ON SCHEMA::Reporting
TO ReportingReader;

ALTER ROLE ReportingReader
ADD MEMBER [ReportingServiceUser];

<table>
    <tr><th>Concept</th><th>What it protects</th><th>Layer</th></tr>
    <tr><td><strong>TLS<strong></td><td>Data in transit between two endpoints</td><td>Network or transport</td></tr>
    <tr><td><strong>TDE / disk encryption<strong></td><td>Data at rest on disk and in backups</td><td>Storage</td></tr>
    <tr><td><strong>Always Encrypted<strong></td><td>Specific columns, even from DBAs, end to end</td><td>Application and database</td></tr>
    <tr><td><strong>VPN / private endpoint<strong></td><td>Which network paths exist at all</td><td>Network</td></tr>
    <tr><td><strong>OAuth / Entra ID<strong></td><td>Who you are and what you may access</td><td>Identity</td></tr>
    <tr><td><strong>mTLS (mutual TLS)<strong></td><td>Both client and server present certificates</td><td>Transport, for service-to-service trust</td></tr>
</table>

# **Leadership and Accountability**

## **How do you prioritize competing production incidents?**

<table>
    <tr><td>I prioritize based on business impact, data integrity, security exposure, number of users affected, and recovery urgency. I communicate the priority and expected next update, coordinate with application and infrastructure teams, and keep a record of decisions. I address the immediate risk first and follow through with root-cause prevention.</td></tr>
</table>

## **Tell us about a time you resolved a difficult technical problem.**

- Use a real example and organize it as <span style = "color:orange">**Situation → Task → Action → Result**</span>. Describe the symptoms, evidence collected, diagnosis, change made, measurable result, and how you prevented recurrence. Do not invent metrics or claim ownership of a project you did not lead.

## **How do you handle disagreements with developers?**

- I focus on evidence and shared outcomes. I review the query, execution plan, application behavior, and operational constraints with the developer. We agree on a testable solution, measure the result, and document the decision. The goal is a reliable application, not proving one team right

## **What would your first 90 days look like?**

# **Questions worth asking**

**What are the most important database reliability or performance challenges you would want this person to address in the first six months?**
- Shows ownership, prioritization, and interest in delivering results.

**Could you describe your SQL Server architecture, particularly how you implement high availability, disaster recovery, and backup validation?**
- Demonstrates awareness of enterprise operational responsibilities.

**How do the DBA and application development teams collaborate on query tuning, schema changes, and production deployments?**
- Particularly relevant because one interviewer is a Senior Developer.

**How are Redgate SQL Source Control and SQL Clone currently used in your development and release workflows?**
- Connects directly to the job description and helps reveal how mature the deployment process is.